# Fine-tune DistilBERT and BERT on GLUE Benchmark Tasks

This notebook performs fine-tuning on GLUE tasks as reported in the DistilBERT paper.
It supports all 9 GLUE tasks with configurable multi-seed runs for evaluation.

The fine-tuning task ran on the JupyterLab on TUWien cluster with duration of ~36h. 
Do not run the notebook with the provided configuration, if you do not have the computational ressources and the time!


## Setup: Install and Import Dependencies

In [1]:
# Install required packages if not already present
import subprocess
import sys

packages_to_install = [
    "transformers>=4.30.0",
    "datasets",
    "accelerate",
    "evaluate",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
]

for package in packages_to_install:
    try:
        __import__(package.split(">=")[0].split("==")[0].replace("-", "_"))
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

C:\Users\Chris\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Installing scikit-learn...


In [2]:
import os
import json
import random
from pathlib import Path
from collections import defaultdict
from typing import Optional, List

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate

# Add project to path
project_root = Path().resolve().parent  #.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from distilbert_repro.utils import set_seed
from distilbert_repro.glue_benchmark import (
    GLUEDataPreprocessor,
    GLUEBenchmarkRunner,
    TaskConfig,
    GLUE_TASK_CONFIG,
)

print("All imports successful")

All imports successful


## Configuration

Edit the settings below to customize benchmark runs.

With the provided configuration all 9 GLUE benchmark task are used to fine-tune BERT and DistilBERT: 5 runs each with different seeds. 



In [15]:
# ============================================================================
# CONFIGURATION - Modify these settings for your experiment
# ============================================================================

# Available GLUE tasks
AVAILABLE_TASKS = list(GLUE_TASK_CONFIG.keys())
print(f"Available GLUE tasks: {AVAILABLE_TASKS}")

# Select which tasks to run
#TASKS_TO_RUN = ["sst2"]
TASKS_TO_RUN = ["sst2", "mrpc", "qqp", "stsb", "cola", "mnli", "qnli", "rte", "wnli"]

# Models to fine-tune 
#MODELS_TO_TRAIN = ["distilbert-base-uncased"]
MODELS_TO_TRAIN = ["distilbert-base-uncased", "bert-base-uncased"]

# Random seeds for multiple runs (median of 5 as per paper)
#SEEDS = [42]
SEEDS = [42, 19, 2005, 2026, 9999]

# Training hyperparameters
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
EVAL_STRATEGY = "epoch"  # Can also be 'steps' with eval_steps parameter
MAX_SEQ_LENGTH = 512

# Output directory for results
OUTPUT_BASE_DIR = Path("./outputs/glue_benchmark")
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration loaded")
print(f"  Tasks:  {TASKS_TO_RUN}")
print(f"  Models: {MODELS_TO_TRAIN}")
print(f"  Seeds:  {SEEDS}")
print(f"  Output: {OUTPUT_BASE_DIR}")


Available GLUE tasks: ['sst2', 'mrpc', 'qqp', 'stsb', 'cola', 'mnli', 'qnli', 'rte', 'wnli']
Configuration loaded
  Tasks:  ['sst2', 'mrpc', 'qqp', 'stsb', 'cola', 'mnli', 'qnli', 'rte', 'wnli']
  Models: ['distilbert-base-uncased', 'bert-base-uncased']
  Seeds:  [42, 19, 2005, 2026, 9999]
  Output: outputs\glue_benchmark


## Step 2: Data Exploration

Explore the SST-2 GLUE task to understand the data structure.

In [ ]:
# Explore the first task
example_task = TASKS_TO_RUN[0]
task_config = TaskConfig.from_task(example_task)

print(f"Exploring task: {example_task.upper()}")
print(f"Task config: {task_config}")

# Load dataset
dataset = load_dataset("glue", task_config.dataset_name)
print(f"\nDataset structure:")
print(dataset)

# Show splits and sizes
for split_name, split_data in dataset.items():
    print(f"\n{split_name}: {len(split_data)} examples")
    if len(split_data) > 0:
        print(f"  Columns: {split_data.column_names}")
        print(f"  First example:")
        for key, value in split_data[0].items():
            print(f"    {key}: {value}")
        break

## Step 3: Fine-tune on GLUE Tasks

Run the full fine-tuning pipeline with multiple seeds.

**!! Note:** This step may take significant time depending on:
- Number of tasks selected
- Number of seeds (default: 5)
- Batch size and number of epochs
- GPU availability

You can reduce time by:
- Running fewer tasks initially (e.g., just ["sst2"])
- Reducing NUM_EPOCHS to 2
- Increasing TRAIN_BATCH_SIZE (if GPU memory allows)

In [ ]:
# Fine-tune all models sequentially and collect results
all_results = {}  # {model_name: raw benchmark results dict}

for MODEL_TO_TRAIN in MODELS_TO_TRAIN:
    print(f"\n{'='*70}")
    print(f"Training model: {MODEL_TO_TRAIN}")
    print(f"  Tasks:  {TASKS_TO_RUN}")
    print(f"  Seeds:  {SEEDS}")
    print(f"  Epochs: {NUM_EPOCHS}  |  LR: {LEARNING_RATE}  |  Batch: {TRAIN_BATCH_SIZE}")
    print(f"{'='*70}")

    runner = GLUEBenchmarkRunner(
        model_name=MODEL_TO_TRAIN,
        output_dir=OUTPUT_BASE_DIR / MODEL_TO_TRAIN.replace("/", "_"),
        seeds=SEEDS,
    )

    results = runner.run_benchmark(
        task_names=TASKS_TO_RUN,
        train_batch_size=TRAIN_BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
    )

    all_results[MODEL_TO_TRAIN] = results
    print(f"\n✓ {MODEL_TO_TRAIN} complete!")

print("\n" + "="*70)
print(f"✓ All {len(MODELS_TO_TRAIN)} models finished!")


In [ ]:
# Quick sanity check – show which tasks each model produced results for
for model_name, res in all_results.items():
    print(f"{model_name}: {list(res.keys())}")


## Step 4: Aggregate Results

Compute median and standard deviation across seeds.

In [ ]:
# Determine metric name for each task
def get_metric_name_for_task(task_name):
    """Get the evaluation metric name for a task."""
    config = TaskConfig.from_task(task_name)
    return f"eval_{config.metric_name}"

# Aggregate results across seeds for every model
all_aggregated = {}  # {model_name: {task_name: stats}}

for model_name, results in all_results.items():
    aggregated = {}
    for task_name in TASKS_TO_RUN:
        metric_name = get_metric_name_for_task(task_name)
        task_agg = GLUEBenchmarkRunner.aggregate_results(
            {task_name: results[task_name]},
            metric_name=metric_name,
        )
        aggregated.update(task_agg)
    all_aggregated[model_name] = aggregated

    # Print per-model summary table
    summary_data = []
    for task_name, stats in aggregated.items():
        summary_data.append({
            "Task":   task_name.upper(),
            "Median": f"{stats['median']:.4f}",
            "Mean":   f"{stats['mean']:.4f}",
            "Std":    f"{stats['std']:.4f}",
            "Min":    f"{stats['min']:.4f}",
            "Max":    f"{stats['max']:.4f}",
        })

    summary_df = pd.DataFrame(summary_data)
    print("="*80)
    print(f"RESULTS — {model_name.upper()}")
    print("="*80)
    print(summary_df.to_string(index=False))

# Side-by-side comparison
print("\n" + "="*80)
print("SIDE-BY-SIDE MEDIAN COMPARISON")
print("="*80)
col_w = 32
header = f"{'Task':<12}" + "".join(f"  {m:<{col_w}}" for m in MODELS_TO_TRAIN)
print(header)
print("-" * len(header))
for task_name in TASKS_TO_RUN:
    row = f"{task_name.upper():<12}"
    for model_name in MODELS_TO_TRAIN:
        score = all_aggregated.get(model_name, {}).get(task_name, {}).get("median", float("nan"))
        row += f"  {score:<{col_w}.4f}"
    print(row)


In [ ]:
# Save aggregated results – one JSON file per model
for model_name, aggregated in all_aggregated.items():
    results_summary = {
        "model": model_name,
        "tasks": TASKS_TO_RUN,
        "seeds": SEEDS,
        "hyperparameters": {
            "train_batch_size": TRAIN_BATCH_SIZE,
            "eval_batch_size":  EVAL_BATCH_SIZE,
            "num_epochs":       NUM_EPOCHS,
            "learning_rate":    LEARNING_RATE,
            "max_seq_length":   MAX_SEQ_LENGTH,
        },
        "results": {
            task_name: {
                "median": float(stats["median"]),
                "mean":   float(stats["mean"]),
                "std":    float(stats["std"]),
                "min":    float(stats["min"]),
                "max":    float(stats["max"]),
                "scores": stats["scores"],
            }
            for task_name, stats in aggregated.items()
        },
    }

    results_path = OUTPUT_BASE_DIR / f"results_{model_name.replace('/', '_')}.json"
    with open(results_path, "w") as f:
        json.dump(results_summary, f, indent=2)
    print(f"✓ Saved: {results_path}")
